# Diabetes Dataset: Cleaning and Exploratory Data Analysis

This notebook profiles the Pima Indians Diabetes dataset and explores data quality, invalid measurements, distributions, and correlations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
DATA_PATH = PROJECT_ROOT / "src/data/diabetes.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## Initial quality checks

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().to_frame("missing_values"))
print(f"Duplicate rows: {df.duplicated().sum()}")
display(df.describe().T)

In [ ]:
zero_counts = (df == 0).sum().sort_values(ascending=False)
display(zero_counts.to_frame("zero_count"))

## Clean invalid measurements

Zero is valid for Pregnancies and Outcome, but not for physiological measurements. The production pipeline applies this rule before median imputation.

In [ ]:
measurement_columns = [
    "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"
]
cleaned_df = df.copy()
median_values = {}
for column in measurement_columns:
    cleaned_df[column] = cleaned_df[column].replace(0, np.nan)
    median_values[column] = cleaned_df[column].median()
    cleaned_df[column] = cleaned_df[column].fillna(median_values[column])
cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)
pd.Series(median_values, name="imputation_median")

In [ ]:
assert cleaned_df.isna().sum().sum() == 0
assert (cleaned_df[measurement_columns] == 0).sum().sum() == 0
assert cleaned_df["Outcome"].isin([0, 1]).all()
display(cleaned_df.head())

## Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=cleaned_df, x="Outcome", ax=axes[0])
axes[0].set_title("Outcome distribution")
sns.histplot(data=cleaned_df, x="Glucose", hue="Outcome", kde=True, ax=axes[1])
axes[1].set_title("Glucose distribution by outcome")
plt.tight_layout()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(cleaned_df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Feature correlation matrix")
plt.tight_layout()

In [ ]:
display(cleaned_df.groupby("Outcome")[measurement_columns + ["BMI", "Age"]].mean().round(2))